# 1. BUSINESS CASE & PROBLEM DEFINITION

## 1.2 BUSINESS UNDERSTANDING  
La rentabilidad de un hotel depende directamente del nivel de ocupación real, es decir, del número de reservas que finalmente se materializan. Sin embargo, una proporción significativa de las reservas termina en cancelaciones. Cuando estas cancelaciones no se compensan con nuevas reservas en un periodo razonable, el hotel pierde ingresos y reduce su eficiencia operativa.

El objetivo de negocio es anticipar qué reservas tienen mayor probabilidad de cancelarse para permitir al hotel tomar decisiones más inteligentes. Con esta información, el equipo de Revenue Management y el departamento de Reservas podrán:

- aplicar estrategias de overbooking controlado,

- ajustar precios de forma dinámica,

- mejorar la comunicación con clientes de alto riesgo,

- adaptar las políticas de cancelación según el perfil de la reserva.

El modelo permitirá minimizar el impacto negativo de las cancelaciones, optimizar la ocupación y mejorar la rentabilidad sin perjudicar la experiencia del cliente.

## 1.2 ML OBJECTIVE
El problema se traduce en un objetivo de Machine Learning claro:

**Tipo de problema:** *Clasificación binaria*

***Variable target:** *is_canceled*

**Clases:**

- 0 → Reserva que llega a término

- 1 → Reserva cancelada

Métricas adecuadas según la guía
Dado que el dataset está desbalanceado, las métricas recomendadas son:
F1-score, Recall (especialmente importante si queremos detectar cancelaciones), Precision, ROC-AUC

Dado que el coste de no detectar una cancelación es mayor que el de una falsa alarma, se priorizará el Recall y el F1-score sobre el Accuracy.

## 1.3 ACTION PLAN

En función de la predicción del modelo:

- Si la reserva tiene alta probabilidad de cancelarse:

- activar estrategias de overbooking controlado,

- enviar comunicaciones preventivas al cliente,

- ajustar precios dinámicamente,

- aplicar políticas de cancelación más estrictas.

Si la probabilidad es baja:

- mantener condiciones estándar,

- priorizar la reserva en caso de alta demanda.

**Umbral mínimo de rendimiento**

El modelo se considerará útil si alcanza un ROC-AUC ≥ 0.75 y un Recall ≥ 0.70 en la clase de cancelación.

**Datos y licencia**
Dataset: Hotel Booking Demand (Kaggle)
Accesible: Sí
Licencia: Open Data (uso permitido para proyectos educativos)

# 2. DATA UNDERSTANDING

## 2.1 FIRST LOOK

In [13]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_csv('src/data_sample/hotel_bookings.csv')

### Target `is_canceled`  

In [15]:
print(f"Shape: {df.shape}")
df.info()
df.describe(include='all')
(df.isnull().mean() * 100).sort_values(ascending=False).round(1)

Shape: (119390, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 

company                           94.3
agent                             13.7
country                            0.4
children                           0.0
arrival_date_month                 0.0
arrival_date_week_number           0.0
hotel                              0.0
is_canceled                        0.0
stays_in_weekend_nights            0.0
arrival_date_day_of_month          0.0
adults                             0.0
stays_in_week_nights               0.0
babies                             0.0
meal                               0.0
lead_time                          0.0
arrival_date_year                  0.0
distribution_channel               0.0
market_segment                     0.0
previous_bookings_not_canceled     0.0
is_repeated_guest                  0.0
reserved_room_type                 0.0
assigned_room_type                 0.0
booking_changes                    0.0
previous_cancellations             0.0
deposit_type                       0.0
days_in_waiting_list     

Porcentaje de nulos por columna:  
children = 0.003%  
country = 0.409%  
agent = 13.686%  
company = 94.306%  

## 2.2 DATA DICTIONARY

La columna `reservation_status` contiene valores equivalentes al target.  
Por otro lado, la columna `reservation_status_date` representa la fecha en la que se registró el estado final de la reserva (cancelación, check-out o no-show). Esta información solo existe después de que la reserva ya ha concluido, por lo que no está disponible en el momento en que queremos predecir si una reserva será cancelada. Usarla introduciría fuga de información (data leakage), ya que el modelo aprendería patrones derivados del resultado final de la reserva.  
En tercer lugar también se descartarían aquellas columnas cuyo porcentaje de nulos sobrepasa el 80% `descartable_null`

In [31]:
df[['reservation_status', 'is_canceled']].groupby('reservation_status').mean()

,is_canceled
reservation_status,
Canceled,1.0
Check-Out,0.0
No-Show,1.0


#### Importancia preliminar de las variables (Valoración inicial)

**Variables con alta aportación esperada**  
- `lead_time`: las reservas con mucha antelación se suelen cancelar más  
- `adr`: 'Tarifa media Diaria' El precio influye en la probabiliad de cancelación  
- `deposit_type`: los depósitos no reembonsables reducen cancelaciones
- `previous_cancellations`: comportamiento histórico del cliente
- `previous_bookings_not_canceled`: Número de reservas anteriores no canceladas por el cliente antes de la reserva actual
- `total_of_special_requests`: los clientes con más requisitos cancelan menos  
- `booking_changes`: Número de modificaciones de la reserva, hasta el check-in o cancelación

**Variables con aportación media**
- `agent`: algunas agencias tienen tasas de cancelación distintas
- `market_segments`: algunso segmenos cancelan más (TA =Travel Agents, TO =Tour Operators)
- `customer_type`: familias, parejas o grupos tiene comportamientos distintos
- `hotel`: tipo de hotel  
- `arrival_date_month  `
- `arrival_date_day_of_month`:habrá que valorarlo en el EDA para ver si hay un patrón
- `arrival_date_week_number`:habrá que valorarlo en el EDA para ver si hay un patrón
- `arrival_date_year`: habrá que valorarlo en el EDA para ver si hay un patrón
- `stays_in_weekend_nights`
- `stays_in_week_nights`
- `adults`
- `children`
- `babies`
- `country`  (si no tiene mucha cardinalidad)

**Variable con aportación baja o redundante**
- `assigned_room_type`
- `reserved_room_type`  
- `meal`  
- `is_repeated_guest`
- `required_car_parking_spaces`: Número de plazas de aparcamiento solicitadas por el huésped
- `arrival_date_week_number`: poca relación con cancelación
- `arrival_date_year`: casi no aporta variabilidad

**Variables descartadas**
- `company`: >80% nulos -> no aporta valor
- `reservation_status`: information leakage (coincide con el target)  
- `reservation_status_date`: information leakage. Previamente explicado


In [34]:
leakage_cols = ['reservation_status', 'reservation_status_date']
null_pct = (df.isnull().mean() * 100).round(1)
descartable_null = null_pct > 80
descartable_manual = df.columns.isin(leakage_cols)

descartable = descartable_null | descartable_manual

data_dict = pd.DataFrame({
    'columna': df.columns,
    'dtype': df.dtypes.values,
    'nulos_pct': null_pct.values,
    'unicos': df.nunique().values,
    'ejemplo': [df[c].dropna().iloc[0] if len(df[c].dropna()) > 0 else None for c in df.columns],
    'valor_modelo': [
        'alta' if col in ['lead_time','adr','deposit_type','previous_cancellations'] else
        'media' if col in ['agent','market_segment','customer_type'] else
        'leakage' if col in ['reservation_status','reservation_status_date'] else
        'irrelevante' if col in ['arrival_date_week_number'] else
        'pendiente'
        for col in df.columns
    ],
    'descartable': descartable.values
})
data_dict

,columna,dtype,nulos_pct,unicos,ejemplo,valor_modelo,descartable
0,hotel,object,0.0,2,Resort Hotel,pendiente,False
1,is_canceled,int64,0.0,2,0,pendiente,False
2,lead_time,int64,0.0,479,342,alta,False
3,arrival_date_year,int64,0.0,3,2015,pendiente,False
4,arrival_date_month,object,0.0,12,July,pendiente,False
5,arrival_date_week_number,int64,0.0,53,27,irrelevante,False
6,arrival_date_day_of_month,int64,0.0,31,1,pendiente,False
7,stays_in_weekend_nights,int64,0.0,17,0,pendiente,False
8,stays_in_week_nights,int64,0.0,35,0,pendiente,False
9,adults,int64,0.0,14,2,pendiente,False


## 2.3 TRAIN-TEST SPLIT

In [36]:
from sklearn.model_selection import train_test_split

TARGET = 'is_canceled'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y  # solo en clasificación
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (95512, 31), Test: (23878, 31)
